<a href="https://colab.research.google.com/github/amtaai/lab-cv-amta/blob/dev%2Fdaniel-v-jepa-2/v-jepa-2/colab/vjepa2_smoke_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Smoke test — encoder V-JEPA 2.1 congelado (Colab)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amtaai/lab-cv-amta/blob/dev/daniel-v-jepa-2/v-jepa-2/colab/vjepa2_smoke_test.ipynb)

**Que valida este notebook** (correr en GPU: `Entorno de ejecucion > Cambiar tipo de entorno > GPU`):

1. Que el checkpoint oficial de V-JEPA 2.1 se descarga y carga (`strict=True`).
2. Que el encoder corre un forward sobre un video real y produce las shapes esperadas
   (`[1, T/2 * (S/16)^2, D]` con tubelet 2 y patch 16).
3. Check **cualitativo** de las dense features 2.1 via PCA (estructura espacial coherente).
4. (Opcional, GPU grande) que las features son semanticamente utiles: probe atentivo SSv2
   pre-entrenado sobre el encoder 2.0 ViT-g 384, igual que el demo oficial de Meta.

**Que NO valida**: nada del lazo de auto-correccion (detector, SAM2, VLM, pseudo-labels) —
eso es `train_pipeline.ipynb` y hoy es scaffold. V-JEPA da *embeddings*, no detecciones.

**Gotcha upstream (verificado 2026-07-02)**: `facebookresearch/vjepa2@main` tiene
`VJEPA_BASE_URL = "http://localhost:8300"` (quedo de testing), asi que
`torch.hub.load(..., pretrained=True)` esta ROTO. Aca se instancia la arquitectura con
`pretrained=False` y el checkpoint se baja a mano de `dl.fbaipublicfiles.com/vjepa2`
(URLs verificadas con HTTP 200 y tamanos reales).

## 1. GPU disponible

In [ ]:
import torch

print("torch:", torch.__version__, "| cuda:", torch.version.cuda)
assert torch.cuda.is_available(), "Sin GPU: Entorno de ejecucion > Cambiar tipo de entorno > GPU"
print("GPU:", torch.cuda.get_device_name(0))
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")

## 2. Repo del proyecto + deps livianas

Se clona `lab-cv-amta` (rama `dev/daniel-v-jepa-2`) solo para usar `core/config.py`
(mapeo tier -> entrypoint). `third_party/` NO viene en el repo (gitignored): el codigo
de V-JEPA 2 lo trae `torch.hub` clonando `facebookresearch/vjepa2`.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/amtaai/lab-cv-amta.git"
BRANCH = "dev/daniel-v-jepa-2"
REPO_DIR = "/content/lab-cv-amta"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, "--single-branch", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/v-jepa-2")

os.environ["AMTA_RUNTIME"] = "colab"
os.environ["AMTA_TIER"] = "edge"  # edge -> 2.1 ViT-L 384 | server -> 2.1 ViT-g 384

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "einops", "decord"], check=True)

from core.config import load_config

cfg = load_config()
print("runtime:", cfg.runtime.value, "| tier:", cfg.tier.value, "| entrypoint del tier:", cfg.vjepa_entrypoint)

## 3. Cargar el encoder (arquitectura via torch.hub + checkpoint manual)

Descargas (elegir segun runtime): 2.1 ViT-B 384 ~1.6 GB (default) · 2.1 ViT-L 384
~4.9 GB · 2.1 ViT-g 384 ~16 GB (requiere `xformers` y un runtime grande).

**Limite del runtime estandar de Colab (T4: ~12.7 GB RAM / 15 GB VRAM / ~112 GB disco,
medido 2026-07-02)**: el state_dict se carga primero a RAM de sistema, asi que los
checkpoints *giant* (~16 GB) NO entran ni para cargarse — en este runtime usar solo
ViT-B o ViT-L. Ambos corren bien en la T4 (fp16 automatico; la T4 no tiene bf16).

In [ ]:
VJEPA_BASE_URL = "https://dl.fbaipublicfiles.com/vjepa2"

# entrypoint -> (archivo de checkpoint, key del state_dict, strict)
# keys/strict replican src/hub/backbones.py del upstream: 2.1 B/L usan "ema_encoder"
# y cargan strict; 2.1 g usa "target_encoder"; 2.0 carga strict=False (pos_embed vs RoPE).
CHECKPOINTS = {
    "vjepa2_1_vit_base_384": ("vjepa2_1_vitb_dist_vitG_384.pt", "ema_encoder", True),
    "vjepa2_1_vit_large_384": ("vjepa2_1_vitl_dist_vitG_384.pt", "ema_encoder", True),
    "vjepa2_1_vit_giant_384": ("vjepa2_1_vitg_384.pt", "target_encoder", True),
    "vjepa2_vit_large": ("vitl.pt", "target_encoder", False),
    "vjepa2_vit_giant_384": ("vitg-384.pt", "target_encoder", False),
}

# None -> usa el entrypoint del tier (cfg.vjepa_entrypoint). Para el primer smoke test
# arrancamos con ViT-B (descarga chica); cuando cierre, probar el del tier.
ENTRYPOINT = "vjepa2_1_vit_base_384"

entrypoint = ENTRYPOINT or cfg.vjepa_entrypoint
ckpt_file, ckpt_key, ckpt_strict = CHECKPOINTS[entrypoint]
print("entrypoint:", entrypoint, "| checkpoint:", ckpt_file)

if "giant" in entrypoint or "gigantic" in entrypoint:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xformers"], check=True)

In [ ]:
# Arquitectura sin pesos (torch.hub clona facebookresearch/vjepa2 en ~/.cache/torch/hub)
encoder, predictor = torch.hub.load("facebookresearch/vjepa2", entrypoint, pretrained=False, trust_repo=True)
del predictor  # el smoke test usa solo el encoder congelado

# Checkpoint real (queda cacheado en ~/.cache/torch/hub/checkpoints)
state = torch.hub.load_state_dict_from_url(f"{VJEPA_BASE_URL}/{ckpt_file}", map_location="cpu")
sd = {k.replace("module.", "").replace("backbone.", ""): v for k, v in state[ckpt_key].items()}
msg = encoder.load_state_dict(sd, strict=ckpt_strict)
print("load_state_dict:", msg)
del state, sd

encoder = encoder.cuda().eval()
for p in encoder.parameters():
    p.requires_grad_(False)
torch.cuda.empty_cache()

n_params = sum(p.numel() for p in encoder.parameters())
print(f"encoder: {type(encoder).__name__} | params: {n_params / 1e6:.0f}M | embed_dim: {encoder.embed_dim}")

## 4. Video de entrada

Default: clip corto de bowling (el mismo del demo oficial). Para probar con un video
propio (p. ej. una zona de mall): subirlo con el icono de carpeta de la izquierda y
apuntar `VIDEO_PATH` al archivo (`/content/mi_video.mp4`).

In [ ]:
import urllib.request

VIDEO_PATH = "sample_video.mp4"
VIDEO_URL = (
    "https://huggingface.co/datasets/nateraw/kinetics-mini/resolve/main/"
    "val/bowling/-WH-lxmGJVY_000005_000015.mp4"
)

if not os.path.exists(VIDEO_PATH):
    urllib.request.urlretrieve(VIDEO_URL, VIDEO_PATH)
print("video:", VIDEO_PATH, f"({os.path.getsize(VIDEO_PATH) / 1e6:.1f} MB)")

## 5. Preprocesamiento (transform de evaluacion del upstream)

Replica `build_pt_video_transform` de `notebooks/vjepa2_demo.ipynb` (upstream), usando
los modulos de transforms del repo que `torch.hub` dejo cacheado. Muestreo uniforme de
64 frames (los checkpoints `fpc64` se entrenaron con clips de 64 frames).

In [ ]:
import glob

import numpy as np
from decord import VideoReader

hub_repo = glob.glob(os.path.join(torch.hub.get_dir(), "facebookresearch_vjepa2_*"))[0]
sys.path.insert(0, hub_repo)
import src.datasets.utils.video.transforms as video_transforms
import src.datasets.utils.video.volume_transforms as volume_transforms

IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
IMG_SIZE = 384 if "384" in entrypoint else 256
NUM_FRAMES = 64


def build_eval_transform(img_size):
    short_side = int(256.0 / 224 * img_size)
    return video_transforms.Compose(
        [
            video_transforms.Resize(short_side, interpolation="bilinear"),
            video_transforms.CenterCrop(size=(img_size, img_size)),
            volume_transforms.ClipToTensor(),
            video_transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ]
    )


vr = VideoReader(VIDEO_PATH)
frame_idx = np.linspace(0, len(vr) - 1, NUM_FRAMES).astype(int)
frames = vr.get_batch(frame_idx).asnumpy()  # T x H x W x C (uint8)
video = torch.from_numpy(frames).permute(0, 3, 1, 2)  # T x C x H x W

clip = build_eval_transform(IMG_SIZE)(video).unsqueeze(0).cuda()  # 1 x C x T x H x W
print("frames origen:", frames.shape, "-> clip:", tuple(clip.shape))

## 6. Forward: embeddings del encoder congelado

Esperado: `[1, (64/2) * (IMG_SIZE/16)^2, embed_dim]` — p. ej. `(1, 18432, 768)` para
2.1 ViT-B 384 (32 tubelets x 24x24 patches).

In [ ]:
import time

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
torch.cuda.reset_peak_memory_stats()

with torch.inference_mode(), torch.autocast("cuda", dtype=dtype):
    encoder(clip)  # warmup
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    feats = encoder(clip)  # 1 x N_tokens x D
    torch.cuda.synchronize()
    dt = time.perf_counter() - t0

n_expected = (NUM_FRAMES // 2) * (IMG_SIZE // 16) ** 2
print(f"features: {tuple(feats.shape)} | esperado: (1, {n_expected}, {encoder.embed_dim})")
assert feats.shape[1] == n_expected and feats.shape[2] == encoder.embed_dim
print(f"forward: {dt * 1000:.0f} ms ({dtype}) | VRAM pico: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
print("SMOKE TEST OK: checkpoint cargado + forward con shapes esperadas")

## 7. Check cualitativo: PCA de las dense features

Proyeccion PCA(3) -> RGB de los patch tokens en tres instantes del clip. Si la mejora
de dense features de 2.1 esta presente, los mapas deberian mostrar estructura espacial
coherente (figura vs fondo, objetos que se mueven). **Ojo**: esto es un check visual
cualitativo, no una deteccion ni una metrica.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

T_TOK, GRID = NUM_FRAMES // 2, IMG_SIZE // 16
fmap = feats.float().reshape(T_TOK, GRID, GRID, -1).cpu().numpy()

t_show = [0, T_TOK // 2, T_TOK - 1]
pca = PCA(n_components=3)
pca.fit(fmap[t_show].reshape(-1, fmap.shape[-1]))

fig, axes = plt.subplots(2, len(t_show), figsize=(4 * len(t_show), 8))
for col, t in enumerate(t_show):
    axes[0, col].imshow(frames[2 * t])  # frame RGB que cae en ese tubelet
    axes[0, col].set_title(f"frame (t_token={t})")
    comp = pca.transform(fmap[t].reshape(-1, fmap.shape[-1])).reshape(GRID, GRID, 3)
    comp = (comp - comp.min()) / (comp.max() - comp.min() + 1e-8)
    axes[1, col].imshow(comp)
    axes[1, col].set_title("PCA(3) de features densas")
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. (Opcional) Probe atentivo SSv2 — check semantico

Replica el demo oficial: encoder 2.0 ViT-g 384 congelado + probe atentivo pre-entrenado
en Something-Something V2 (174 clases de accion). Con el clip de bowling, el top-5
deberia incluir algo tipo *"Putting [something] into [something]"*.

Requiere descargar `vitg-384.pt` (~15.7 GB) + probe (357 MB) e instalar `xformers`.
**NO viable en el runtime estandar de Colab**: el checkpoint no entra en ~12.7 GB de
RAM de sistema. Solo con runtime high-RAM + GPU grande (Colab Pro: A100/L4 con RAM
ampliada). Por eso viene gateado con `RUN_PROBE = False`.

In [ ]:
RUN_PROBE = False  # poner True solo con GPU grande (descarga ~16 GB)

if RUN_PROBE:
    import json

    import torch.nn.functional as F

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xformers"], check=True)
    from src.models.attentive_pooler import AttentiveClassifier

    enc_g, _ = torch.hub.load("facebookresearch/vjepa2", "vjepa2_vit_giant_384", pretrained=False, trust_repo=True)
    state = torch.hub.load_state_dict_from_url(f"{VJEPA_BASE_URL}/vitg-384.pt", map_location="cpu")
    sd = {k.replace("module.", "").replace("backbone.", ""): v for k, v in state["target_encoder"].items()}
    enc_g.load_state_dict(sd, strict=False)
    del state, sd
    enc_g = enc_g.cuda().eval()

    probe = AttentiveClassifier(embed_dim=enc_g.embed_dim, num_heads=16, depth=4, num_classes=174).cuda().eval()
    pstate = torch.hub.load_state_dict_from_url(f"{VJEPA_BASE_URL}/evals/ssv2-vitg-384-64x2x3.pt", map_location="cpu")
    psd = {k.replace("module.", ""): v for k, v in pstate["classifiers"][0].items()}
    probe.load_state_dict(psd, strict=False)
    del pstate, psd

    urllib.request.urlretrieve(
        "https://huggingface.co/datasets/huggingface/label-files/resolve/"
        "d79675f2d50a7b1ecf98923d42c30526a51818e2/something-something-v2-id2label.json",
        "ssv2_classes.json",
    )
    classes = json.load(open("ssv2_classes.json"))

    clip384 = build_eval_transform(384)(video).unsqueeze(0).cuda()
    with torch.inference_mode():
        logits = probe(enc_g(clip384))
    top5 = logits.topk(5)
    probs = F.softmax(top5.values[0], dim=-1) * 100  # % relativo dentro del top-5
    for i, p in zip(top5.indices[0], probs):
        print(f"{classes[str(i.item())]:<60s} {p:.1f}%")

## 9. Resultado y proximos pasos

Si las secciones 1-6 corrieron sin error, el smoke test paso: el encoder V-JEPA 2.1
carga sus pesos oficiales y produce embeddings con las shapes correctas en Colab.

Proximos pasos en el proyecto:

1. Implementar `core/perception/encoder.py::VJepa2Encoder` usando exactamente este
   camino de carga (pretrained=False + checkpoint manual) — el `pretrained=True` del
   hub sigue roto upstream.
2. Sumar detector open-vocab + SAM2 en `train_pipeline.ipynb` (lazo completo).
3. Repetir este smoke test con un clip del dominio real (mall) y el tier del config.